# SupportIQ — Stage 1.2: Data Profiling, Token Counting & Template Clusters

> **Welcome to Data Profiling!**
> Before we fine-tune any AI model, we need to answer three practical questions:
> 1. **Classes:** Are categories and intents balanced, or are some missing?
> 2. **Length (Block 7):** How many tokens are in customer questions and agent answers? (Tells us our `max_seq_length`).
> 3. **Templates (Blocks 8 & 9):** How many customer questions are just slight paraphrases of each other? (Prevents data leakage).


### 1. Setup & Load Data
First, we import our libraries and load the raw dataset saved in Stage 1.1 (`data/raw/bitext_raw.parquet`).


In [1]:
import sys
from pathlib import Path

# Add project root to sys.path so we can import supportiq
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
if str(project_root / "src") not in sys.path:
    sys.path.insert(0, str(project_root / "src"))

import polars as pl

from supportiq.data.load import load_raw_dataframe

# Load the 26,872 raw rows
df = load_raw_dataframe()
print(f"Dataset loaded successfully: {df.height:,} rows.")

[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


Dataset loaded successfully: 26,872 rows.


### 2. Category & Intent Balance
We count how many examples exist for each category and customer intent.


In [2]:
# Count rows per category
category_counts = (
    df.group_by("category")
    .agg(pl.len().alias("count"))
    .with_columns((pl.col("count") / df.height * 100).round(1).alias("percent"))
    .sort("count", descending=True)
)
print("Categories:")
print(category_counts)

# Summary of unique classes
print(f"\nTotal Categories: {df['category'].n_unique()}")
print(f"Total Intents:    {df['intent'].n_unique()}")

Categories:
shape: (11, 3)
┌──────────────┬───────┬─────────┐
│ category     ┆ count ┆ percent │
│ ---          ┆ ---   ┆ ---     │
│ str          ┆ u32   ┆ f64     │
╞══════════════╪═══════╪═════════╡
│ ACCOUNT      ┆ 5986  ┆ 22.3    │
│ ORDER        ┆ 3988  ┆ 14.8    │
│ REFUND       ┆ 2992  ┆ 11.1    │
│ CONTACT      ┆ 1999  ┆ 7.4     │
│ INVOICE      ┆ 1999  ┆ 7.4     │
│ …            ┆ …     ┆ …       │
│ FEEDBACK     ┆ 1997  ┆ 7.4     │
│ DELIVERY     ┆ 1994  ┆ 7.4     │
│ SHIPPING     ┆ 1970  ┆ 7.3     │
│ SUBSCRIPTION ┆ 999   ┆ 3.7     │
│ CANCEL       ┆ 950   ┆ 3.5     │
└──────────────┴───────┴─────────┘

Total Categories: 11
Total Intents:    27


### 3. Check Template Slot Placeholders
In customer support, templates have slots like `{{Order Number}}` or `{{Account ID}}`.
We count how many distinct slots exist so we don't accidentally delete them later.


In [3]:
import re

# Find any text inside double curly braces: {{...}}
pattern = re.compile(r"\{\{([^}]+)\}\}")

detected_slots = set()
for text in df["instruction"].to_list() + df["response"].to_list():
    matches = pattern.findall(text)
    for slot_name in matches:
        detected_slots.add(slot_name)

print(f"Found {len(detected_slots)} unique slot types in the dataset.")
print("5 Example slots:")
for example in sorted(detected_slots)[:5]:
    print(f"  - {{{{{example}}}}}")

Found 391 unique slot types in the dataset.
5 Example slots:
  - {{Access Key}}
  - {{Access Key Recovery}}
  - {{Access Key Reset Page URL}}
  - {{Access Key Retrieval}}
  - {{Account}}


### 4. [BLOCK 7] Token Length Profiling with Qwen Tokenizer

#### What is a "Token"?
AI models (LLMs) cannot read plain words. Instead, a **Tokenizer** splits text into chunks of characters and turns each chunk into a number (a Token ID).
- For example: `"cancel my order"` might turn into 3 numbers: `[2340, 852, 1980]`.

#### Why do we measure token lengths?
When fine-tuning, every training example must fit into the model's memory (`max_seq_length`).
- If our messages are only ~120 tokens, setting `max_seq_length = 512` is plenty.
- If we didn't check, we might blindly guess `2048`, which uses 4x more GPU memory and costs 4x more!


In [4]:
import numpy as np
from transformers import AutoTokenizer

# 1. Load the official Qwen tokenizer (tiny download, runs on CPU)
print("Loading Qwen tokenizer...")
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B")

# 2. Extract customer instructions and agent responses as plain Python lists
all_instructions = df["instruction"].to_list()
all_responses = df["response"].to_list()

# 3. Convert texts into tokens and count how many tokens each sentence has
# (We do this step by step so it is easy to read)
instruction_token_counts = []
response_token_counts = []
total_sequence_lengths = []

print("Counting tokens across all 26,872 rows...")
for inst, resp in zip(all_instructions, all_responses, strict=True):
    # Count tokens in customer instruction
    num_inst_tokens = len(tokenizer.encode(inst, add_special_tokens=False))
    # Count tokens in assistant response
    num_resp_tokens = len(tokenizer.encode(resp, add_special_tokens=False))

    # Combined length (user prompt + assistant answer)
    num_total_tokens = num_inst_tokens + num_resp_tokens

    instruction_token_counts.append(num_inst_tokens)
    response_token_counts.append(num_resp_tokens)
    total_sequence_lengths.append(num_total_tokens)

# 4. Print clean, easy-to-understand statistics
print("\n=== TOKEN LENGTH RESULTS ===")
print(
    f"Customer Questions:  median {np.median(instruction_token_counts):.0f} tokens (max {np.max(instruction_token_counts)} tokens)"
)
print(
    f"Assistant Answers:   median {np.median(response_token_counts):.0f} tokens (max {np.max(response_token_counts)} tokens)"
)
print(
    f"Total Sequence:      median {np.median(total_sequence_lengths):.0f} tokens (max {np.max(total_sequence_lengths)} tokens)"
)

# 5. Check if any message is longer than 512 tokens
cut_off_count = sum(1 for length in total_sequence_lengths if length > 512)
print(
    f"\nMessages cut off at max_seq_length=512: {cut_off_count} out of {len(total_sequence_lengths)} (0.00% loss)"
)
print("=> CONCLUSION: max_seq_length=512 is 100% safe for fine-tuning!")

Loading Qwen tokenizer...


Counting tokens across all 26,872 rows...



=== TOKEN LENGTH RESULTS ===
Customer Questions:  median 10 tokens (max 24 tokens)
Assistant Answers:   median 104 tokens (max 478 tokens)
Total Sequence:      median 115 tokens (max 490 tokens)

Messages cut off at max_seq_length=512: 0 out of 26872 (0.00% loss)
=> CONCLUSION: max_seq_length=512 is 100% safe for fine-tuning!


### 5. [BLOCK 8] Finding Near-Duplicate Templates (MinHash-LSH)

#### What is this problem?
In the Bitext dataset, many customer questions are just slight variations of each other:
- Sentence A: *"cancel order {{Order Number}}"*
- Sentence B: *"how do I cancel order {{Order Number}}?"*
- Sentence C: *"please cancel order {{Order Number}}"*

If Sentence A goes into the **Training Set** and Sentence B goes into the **Test Set**, the model won't learn to understand English—it will just memorize the template!

#### How does MinHash-LSH work? (Step by Step)
1. **Shingling (3-word chunks):** Break a sentence into 3-word overlapping groups:
   - *"please cancel order now"* -> `{"please cancel order", "cancel order now"}`.
2. **MinHash Fingerprint:** Convert these word chunks into a small numeric fingerprint.
3. **LSH Index (Bucket Matching):** Put fingerprints into buckets. Sentences that share 80% or more of the same 3-word chunks land in the **same cluster bucket**.


In [5]:
from datasketch import MinHash, MinHashLSH


# Step 1: Helper function to break a sentence into 3-word chunks ("shingles")
def make_3word_chunks(text: str) -> set[str]:
    words = text.lower().split()
    if len(words) < 3:
        return {text.lower()}  # If sentence is too short, return the whole sentence
    chunks = set()
    for i in range(len(words) - 2):
        chunk = " ".join(words[i : i + 3])
        chunks.add(chunk)
    return chunks


# Step 2: Initialize the LSH matcher (80% similarity threshold)
lsh_index = MinHashLSH(threshold=0.80, num_perm=128)

# Step 3: Take a sample of 500 instructions to demonstrate how clusters form
sample_instructions = df["instruction"][:500].to_list()
fingerprints = []

print("Creating fingerprints for 500 sample instructions...")
for row_idx, instruction_text in enumerate(sample_instructions):
    # Create MinHash fingerprint
    m = MinHash(num_perm=128)
    for chunk in make_3word_chunks(instruction_text):
        m.update(chunk.encode("utf8"))

    # Store fingerprint and insert into index
    fingerprints.append(m)
    lsh_index.insert(f"row_{row_idx}", m)

print("LSH Index built successfully!")

Creating fingerprints for 500 sample instructions...
LSH Index built successfully!


### 6. [BLOCK 9] Inspecting Real Template Clusters

Now that the index is built, we can take a question and ask:
> *"Which other questions in the dataset are near-duplicate paraphrases of this question?"*


In [6]:
# Let's take the first question (Row 0) and search for its paraphrases
seed_row_id = 0
seed_question = sample_instructions[seed_row_id]
seed_fingerprint = fingerprints[seed_row_id]

# Query the LSH index for matches
matching_row_keys = lsh_index.query(seed_fingerprint)

print(f"Seed Question (Row {seed_row_id}):")
print(f"  -> '{seed_question}'")
print(f"\nFound {len(matching_row_keys)} near-duplicate paraphrases in this cluster:")

for key in sorted(matching_row_keys):
    matched_row_num = int(key.replace("row_", ""))
    print(f"  - [Row {matched_row_num:03d}]: {sample_instructions[matched_row_num]}")

print("\n" + "=" * 70)
print("CRITICAL TAKEAWAY FOR STAGE 2:")
print("All of the rows above are variations of the SAME template.")
print("When we split our data (80% train, 10% test), we MUST keep all rows")
print("from the same cluster in the SAME split, so none leak into test!")
print("=" * 70)

Seed Question (Row 0):
  -> 'question about cancelling order {{Order Number}}'

Found 3 near-duplicate paraphrases in this cluster:
  - [Row 000]: question about cancelling order {{Order Number}}
  - [Row 130]: have a question about cancelling order {{Order Number}}
  - [Row 053]: have a question about cancelling order {{Order Number}}

CRITICAL TAKEAWAY FOR STAGE 2:
All of the rows above are variations of the SAME template.
When we split our data (80% train, 10% test), we MUST keep all rows
from the same cluster in the SAME split, so none leak into test!
